# Session 1, PyTorch

Three blocks: environment, math (on paper, typed up in derivations/), build step zero.
Code is typed by hand. Run 2026-11-09 on CPU, torch 2.15.0+cpu.

## Block 1, setup (15 min)

Confirm torch imports, check the device, run one tiny training loop end to end.

In [1]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
random.seed(0)
print("torch", torch.__version__)
print("device", "cuda" if torch.cuda.is_available() else "cpu")

torch 2.15.0+cpu
device cpu


In [2]:
# One tiny training loop, end to end: fit y = 2x + 1.
x = torch.randn(64, 1)
y = 2 * x + 1
lin = nn.Linear(1, 1)
opt = torch.optim.SGD(lin.parameters(), lr=0.1)
for step in range(20):
    opt.zero_grad()
    loss = ((lin(x) - y) ** 2).mean()
    loss.backward()
    opt.step()
    if step % 5 == 0 or step == 19:
        print(f"step {step:2d} loss {loss.item():.6f}")
print(f"learned weight {lin.weight.item():.4f} bias {lin.bias.item():.4f} (true: 2.0, 1.0)")

step  0 loss 3.876311
step  5 loss 0.284889
step 10 loss 0.021037
step 15 loss 0.001566
step 19 loss 0.000198
learned weight 1.9933 bias 0.9923 (true: 2.0, 1.0)


## Block 2, math (30 min, paper first)

Roofline (Scaling Book Ch. 1), transformer FLOPs and the 6N rule, memory accounting.
Written up in ../derivations/session-01-roofline-and-flops.md

## Block 3, build step zero (45 min)

Dataset: addition up to 3 digits. Vocabulary: digits, space, +, =.
Tokenizer for that alphabet. Model skeleton in PyTorch. No training yet.

In [3]:
# Dataset: addition up to 3 digits, e.g. "123+456=579".
def make_example():
    a = random.randint(0, 999)
    b = random.randint(0, 999)
    return f"{a}+{b}={a+b}"

examples = [make_example() for _ in range(5)]
print("sample examples:", examples)

sample examples: ['864+394=1258', '776+911=1687', '430+41=471', '265+988=1253', '523+497=1020']


In [4]:
# Tokenizer for the tiny alphabet: digits, '+', '=', plus pad/bos/eos.
CHARS = list("0123456789+=")
SPECIALS = ["<pad>", "<bos>", "<eos>"]
VOCAB = CHARS + SPECIALS
stoi = {c: i for i, c in enumerate(VOCAB)}
itos = {i: c for c, i in stoi.items()}
print("vocab size:", len(VOCAB), "vocab:", VOCAB)

def encode(s, max_len=16):
    ids = [stoi["<bos>"]] + [stoi[c] for c in s] + [stoi["<eos>"]]
    ids = ids + [stoi["<pad>"]] * (max_len - len(ids))
    return ids[:max_len]

def decode(ids):
    return "".join(itos[i] for i in ids if itos[i] not in ("<pad>", "<bos>", "<eos>"))

enc = encode("123+456=579")
print("encode('123+456=579'):", enc)
print("decode back:", repr(decode(enc)))

vocab size: 15 vocab: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '+', '=', '<pad>', '<bos>', '<eos>']
encode('123+456=579'): [13, 1, 2, 3, 10, 4, 5, 6, 11, 5, 7, 9, 14, 12, 12, 12]
decode back: '123+456=579'


In [5]:
# Transformer skeleton, decoder-only, ~10M params.
class Block(nn.Module):
    def __init__(self, d_model, n_head, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_head, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )

    def forward(self, x, mask):
        h = self.ln1(x)
        a, _ = self.attn(h, h, h, attn_mask=mask, need_weights=False)
        x = x + a
        x = x + self.mlp(self.ln2(x))
        return x

class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=384, n_layer=6, n_head=6, d_ff=1536, max_len=32):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_head, d_ff) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.max_len = max_len

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)[None, :, :]
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=idx.device), diagonal=1)
        for blk in self.blocks:
            x = blk(x, mask)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=stoi["<pad>"])
        return logits, loss

model = TinyTransformer(len(VOCAB))
n_params = sum(p.numel() for p in model.parameters())
print(f"model params: {n_params:,} (~{n_params/1e6:.1f}M)")

# One forward pass on a real batch, to prove the skeleton runs.
batch = torch.tensor([encode(make_example()) for _ in range(4)])
logits, loss = model(batch, targets=batch)
print("batch shape:", tuple(batch.shape), "logits shape:", tuple(logits.shape))
print(f"initial loss (untrained): {loss.item():.4f} (random guessing over {len(VOCAB)} tokens would be ln({len(VOCAB)}) = {__import__('math').log(len(VOCAB)):.4f})")

model params: 10,671,360 (~10.7M)
batch shape: (4, 16) logits shape: (4, 16, 15)
initial loss (untrained): 3.0513 (random guessing over 15 tokens would be ln(15) = 2.7081)


## Done when

Working notebook, one page of checked derivations, dataset + tokenizer + skeleton, and the session write-up in ../writeups/session-01-writeup.md

Done: notebook runs end to end (5/5 code cells executed, outputs above). Derivations in ../derivations/session-01-roofline-and-flops.md. Write-up in ../writeups/session-01-writeup.md. Training is Session 2.